In [1]:
%load_ext autoreload
%autoreload 2

import open3d as o3d
import numpy as np
import os
import sys
from glob import glob
import pickle

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')
from utils.write_output import write_array2txt

from cylinder_extraction.cylinder import Cylinder
from registration.utils import multiway_load_point_clouds as load_point_clouds
from registration.utils import multiway_full_registration as full_registration
from registration.utils import pairwise_registration, combine_point_cloud, multiway_filter_point_clouds

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Input
The first part of the tutorial code reads three point clouds from files. The point clouds are downsampled and visualized together. They are misaligned.

In [2]:
voxel_size = 0.02
data_dir = 'D:/REGISTRATION DATA/External/JB-453 60ft EXT/tank_1'
tank_name = '1'
# pcds_down, pcds_full = load_point_clouds(data_dir,voxel_size)

In [3]:
root_dir = 'D:/REGISTRATION DATA/External/JB-453 60ft EXT'
with open(os.path.join(root_dir, 'tank_info.pkl'), 'rb') as f:
    tank_info = pickle.load(f)
all_cylinders = tank_info['tanks'][tank_name]


In [4]:
in_cyls, out_cyls = multiway_filter_point_clouds(all_cylinders)

### Read point clouds for inlier cylinders

In [5]:
pcds_down, pcds_full = load_point_clouds(in_cyls)

D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9001.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9002.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9003.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9004.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9005.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9006.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9007.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9008.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9009.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9010.txt
D:\REGISTRATION DATA\External\JB-453 60ft EXT\tank_1\9011.txt


## Pose graph

In [6]:
print("Full registration ...")
max_correspondence_distance_coarse = 0.05#voxel_size * 15
max_correspondence_distance_fine = 0.01#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration(pcds_down,
                                   max_correspondence_distance_coarse,
                                   max_correspondence_distance_fine)

Full registration ...
Apply point-to-plane ICP
Coarse alignment
[Open3D DEBUG] ICP Iteration #0: Fitness 0.0170, RMSE 0.0314
[Open3D DEBUG] Residual : 7.32e-04 (# of elements : 10612)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.0174, RMSE 0.0316
[Open3D DEBUG] Residual : 7.44e-04 (# of elements : 10887)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.0181, RMSE 0.0319
[Open3D DEBUG] Residual : 7.63e-04 (# of elements : 11278)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.0188, RMSE 0.0320
[Open3D DEBUG] Residual : 7.75e-04 (# of elements : 11723)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.0198, RMSE 0.0321
[Open3D DEBUG] Residual : 7.84e-04 (# of elements : 12387)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.0210, RMSE 0.0319
[Open3D DEBUG] Residual : 7.81e-04 (# of elements : 13122)
[Open3D DEBUG] ICP Iteration #6: Fitness 0.0227, RMSE 0.0317
[Open3D DEBUG] Residual : 7.76e-04 (# of elements : 14188)
[Open3D DEBUG] ICP Iteration #7: Fitness 0.0241, RMSE 0.0307
[Open3D DEBUG] Residual : 7.14e-04 

Open3D uses the function `global_optimization` to perform pose graph optimization. Two types of optimization methods can be chosen: `GlobalOptimizationGaussNewton` or `GlobalOptimizationLevenbergMarquardt`. The latter is recommended since it has better convergence property. Class `GlobalOptimizationConvergenceCriteria` can be used to set the maximum number of iterations and various optimization parameters.

Class `GlobalOptimizationOption` defines a couple of options. `max_correspondence_distance` decides the correspondence threshold. `edge_prune_threshold` is a threshold for pruning outlier edges. `reference_node` is the node id that is considered to be the global space.

In [7]:
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)

Optimizing PoseGraph ...
[Open3D DEBUG] Validating PoseGraph - finished.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 11 nodes and 55 edges.
[Open3D DEBUG] Line process weight : 7.168860
[Open3D DEBUG] [Initial     ] residual : 1.857537e+03, lambda : 3.984496e+03
[Open3D DEBUG] [Iteration 00] residual : 8.507618e+01, valid edges : 33, time : 0.000 sec.
[Open3D DEBUG] [Iteration 01] residual : 8.266399e+01, valid edges : 34, time : 0.000 sec.
[Open3D DEBUG] [Iteration 02] residual : 8.107701e+01, valid edges : 34, time : 0.000 sec.
[Open3D DEBUG] [Iteration 03] residual : 8.001055e+01, valid edges : 34, time : 0.000 sec.
[Open3D DEBUG] [Iteration 04] residual : 7.930323e+01, valid edges : 34, time : 0.000 sec.
[Open3D DEBUG] [Iteration 05] residual : 7.839867e+01, valid edges : 35, time : 0.000 sec.
[Open3D DEBUG] [Iteration 06] residual : 7.690363e+01, valid edges : 36, time : 0.000 sec.
[Open3D DEBUG] [Iteration 07] residual : 7.501347e+01, valid edges : 36, time 

The global optimization performs twice on the pose graph. The first pass optimizes poses for the original pose graph taking all edges into account and does its best to distinguish false alignments among uncertain edges. These false alignments have small line process weights, and they are pruned after the first pass. The second pass runs without them and produces a tight global alignment. In this example, all the edges are considered as true alignments, hence the second pass terminates immediately.

## Visualize optimization
The transformed point clouds are listed and visualized using `draw_geometries`.

In [ ]:
print("Transform points and display")
for point_id in range(len(pcds_down)):
    pcds_down[point_id].transform(pose_graph.nodes[point_id].pose)
o3d.visualization.draw_geometries(pcds_down,
                                  zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])

## Make a combined point cloud
`PointCloud` has a convenience operator `+` that can merge two point clouds into a single one. In the code below, the points are uniformly resampled using `voxel_down_sample` after merging. This is recommended post-processing after merging point clouds since it can relieve duplicated or over-densified points.

In [8]:
inliner_pcd_combined = combine_point_cloud(pcds_full, pose_graph)
# pcds = load_point_clouds(path_,voxel_size)
# pcd_combined = o3d.geometry.PointCloud()
# pcd_full = np.empty(shape=[0,4])
# for point_id in range(len(pcds)):
#     pcds[point_id].transform(pose_graph.nodes[point_id].pose)
#     A = np.asarray(pcds[point_id].points)
#     B = np.c_[ A, np.ones( A.shape[0] )*point_id ]
#     pcd_full = np.r_[pcd_full,B]

In [9]:
write_array2txt(
    os.path.join(data_dir, f'tank_{tank_name}_inliers_combined_with_ids.txt'),
    inliner_pcd_combined)
write_array2txt(
    os.path.join(data_dir, f'tank_{tank_name}_inliers_combined.txt'),
    inliner_pcd_combined[:,:3])


## Handle outlier cylinders

In [ ]:
cyl_combined = Cylinder(
    metadata = {
        'ptc_filepath': os.path.join(data_dir, 'inliers_combined.txt'),
        'tank_id': 0,
    }
)
out_cyls.insert(0,cyl_combined)

In [ ]:
# read points
pcds_down, pcds_full = load_point_clouds(out_cyls)
# pose graph
print("Full registration ...")
max_correspondence_distance_coarse = 0.2#voxel_size * 15
max_correspondence_distance_fine = 0.02#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration(pcds_down,
                                   max_correspondence_distance_coarse,
                                   max_correspondence_distance_fine)
# optimize
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)
# make combined
pose_graph.nodes.pop(0)
outlier_pcd_combined = combine_point_cloud(pcds_full[1:], pose_graph)
# inlier + outlier
pcd_combined = np.r_[inliner_pcd_combined, outlier_pcd_combined]
# pcd_combined = combine_point_cloud(pcds_full, pose_graph)
# write
write_array2txt(
    data_dir + '\complete_combined_with_ids.txt',
    pcd_combined)